# 🧠 DAE-SSL : Détection d'anomalies cérébrales — OASIS-3

**Objectif** : Entraîner un *Denoising Autoencoder* (DAE) en mode **auto-supervisé** (SSL)  
sur les features FreeSurfer d'OASIS-3, puis utiliser l'**erreur de reconstruction**  
comme score d'anomalie pour distinguer CN / MCI / AD.  

---
| Bloc | Contenu |
|------|--------------------------------------------------|
| 0 | Installation & imports |
| 1 | Chargement des données OASIS-3 |
| 2 | Prétraitement & bruit |
| 3 | Architecture DAE (PyTorch) |
| 4 | Entraînement SSL |
| 5 | Score d'anomalie |
| 6 | Visualisations & évaluation |

---
> **Dataset** : OASIS-3 — FreeSurfer stats (volumes ROI, épaisseurs corticales)  
> **Runtime recommandé** : GPU T4 (Colab gratuit)  
> **Aucun label utilisé pendant l'entraînement** — évaluation seulement

---
## ⚙️ Bloc 0 — Installation & Imports

In [ ]:
# ── Installation des dépendances ──────────────────────────────────────────────
# nilearn  : accès aux datasets neuroimaging publics (dont OASIS)
# pandas   : manipulation CSV
# seaborn  : visualisations statistiques
# scikit-learn : normalisation, métriques
!pip install -q nilearn pandas seaborn scikit-learn torch

In [ ]:
# ── Imports principaux ────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.manifold import TSNE

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ── Configuration globale ─────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Utilise le GPU si disponible (T4 sur Colab)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

# Style des graphiques
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

---
## 📂 Bloc 1 — Chargement des données OASIS-3

**OASIS-3** contient les stats FreeSurfer de ~1000 sujets (CN, MCI, AD).  
On utilise le fichier CSV des volumes/épaisseurs corticaux extrait par FreeSurfer.  

**Option A** : Dataset synthétique (test rapide, pas d'accès OASIS requis)  
**Option B** : Vrai fichier OASIS-3 uploadé sur Colab  

> 👉 Mets `USE_SYNTHETIC = False` et upload ton CSV OASIS pour passer en mode réel.

In [ ]:
# ── Configuration : choisir la source de données ──────────────────────────────

USE_SYNTHETIC = True          # True  → données simulées (aucun accès OASIS requis)
                               # False → fichier CSV réel OASIS-3

OASIS_CSV_PATH = "/content/oasis3_freesurfer_stats.csv"  # chemin si USE_SYNTHETIC=False

# Colonnes attendues dans le CSV réel :
#   - features  : volumes ROI (ex. lh_hippocampus_volume, rh_entorhinal_thickness…)
#   - label col : colonne avec CN / MCI / AD (uniquement pour évaluation)
LABEL_COL = "dx_group"        # nom de la colonne de diagnostic dans ton CSV

In [ ]:
# ── Option A : Génération de données synthétiques ─────────────────────────────
#
# On simule 3 groupes avec des distributions différentes :
#   - CN  (contrôle normal)   : features ~ N(0, 1)
#   - MCI (mild cognitif imp.) : features ~ N(-0.3, 1.1)  légère déviation
#   - AD  (Alzheimer)         : features ~ N(-0.7, 1.3)  forte déviation

if USE_SYNTHETIC:

    N_FEATURES  = 90    # nombre de features FreeSurfer simulées
    N_CN        = 400   # sujets Contrôle Normal
    N_MCI       = 200   # sujets MCI
    N_AD        = 150   # sujets Alzheimer

    rng = np.random.default_rng(SEED)

    # Génération des features par groupe
    X_cn  = rng.normal(loc=0.0,  scale=1.0, size=(N_CN,  N_FEATURES))
    X_mci = rng.normal(loc=-0.3, scale=1.1, size=(N_MCI, N_FEATURES))
    X_ad  = rng.normal(loc=-0.7, scale=1.3, size=(N_AD,  N_FEATURES))

    # Assemblage en DataFrame
    X_all = np.vstack([X_cn, X_mci, X_ad])
    y_all = np.array(["CN"] * N_CN + ["MCI"] * N_MCI + ["AD"] * N_AD)

    feature_names = [f"feature_{i:03d}" for i in range(N_FEATURES)]
    df = pd.DataFrame(X_all, columns=feature_names)
    df["dx_group"] = y_all

    print(f"Dataset synthétique créé : {df.shape}")
    print(df["dx_group"].value_counts())
    df.head(3)

In [ ]:
# ── Option B : Chargement du vrai fichier OASIS-3 ─────────────────────────────
#
# 1) Va sur https://www.oasis-brains.org/ → inscris-toi → télécharge
#    le fichier freesurfer_stats.csv (ou équivalent)
# 2) Dans Colab : panneau gauche → icône dossier → upload le CSV
# 3) Passe USE_SYNTHETIC = False dans la cellule précédente

if not USE_SYNTHETIC:

    df = pd.read_csv(OASIS_CSV_PATH)
    print(f"Fichier chargé : {df.shape}")
    print(f"Colonnes : {list(df.columns[:10])} ...")
    print(df[LABEL_COL].value_counts())
    df.head(3)

---
## 🔍 Bloc 1b — EDA : Visualisation du dataset brut

Avant tout prétraitement, on explore :
1. **Distribution des groupes** (équilibre CN/MCI/AD)
2. **Statistiques descriptives** par groupe
3. **Distributions des features** (histogrammes, outliers)
4. **Corrélations** entre features
5. **Heatmap NaN** (valeurs manquantes)

> Ces visualisations guident les choix de prétraitement (normalisation, imputation, sélection de features).

In [ ]:
# feature_cols défini ici pour l'EDA — réutilisé dans le Bloc 2
feature_cols = [c for c in df.columns if c != "dx_group"]

# ── 1 : Distribution des groupes diagnostiques ───────────────────────────────
#
# Premier check : est-ce que le dataset est équilibré ?
# Un déséquilibre CN >> AD est normal (reflet de la population réelle)
# mais important à connaître pour interpréter les métriques

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

group_counts = df["dx_group"].value_counts()
colors_bar   = ["#4C72B0", "#DD8452", "#C44E52"]

# Barplot
ax = axes[0]
bars = ax.bar(group_counts.index, group_counts.values,
              color=colors_bar[:len(group_counts)], edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, group_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(val), ha="center", va="bottom", fontsize=11)
ax.set_title("Nombre de sujets par groupe")
ax.set_xlabel("Diagnostic")
ax.set_ylabel("Nombre de sujets")
ax.set_ylim(0, group_counts.max() * 1.15)

# Pie chart
ax = axes[1]
ax.pie(group_counts.values, labels=group_counts.index,
       autopct="%1.1f%%", colors=colors_bar[:len(group_counts)],
       startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1})
ax.set_title("Proportion des groupes")

plt.suptitle("Distribution des groupes diagnostiques", fontsize=13)
plt.tight_layout()
plt.show()

print(group_counts)
print(f"Total sujets : {len(df)}  |  Features : {len(feature_cols)}")


In [ ]:
# ── 2 : Statistiques descriptives par groupe ─────────────────────────────────
#
# On compare mean ± std des features entre CN, MCI, AD
# Les features avec grande différence inter-groupe = features discriminantes

print("Statistiques globales (toutes features) :")
print(df[feature_cols].describe().round(3).T.head(10))

print("\nMoyenne par groupe (10 premières features) :")
print(df.groupby("dx_group")[feature_cols[:10]].mean().round(3).T)


In [ ]:
# ── 3 : Distributions des features (avant normalisation) ─────────────────────
#
# On affiche les 12 premières features en histogramme superposé par groupe
# Objectif : voir si les features séparent visuellement CN vs AD
#            et si les distributions sont normales ou skewed

N_FEATURES_SHOW = min(12, len(feature_cols))
cols_show       = feature_cols[:N_FEATURES_SHOW]
COLOR_MAP       = {"CN": "#4C72B0", "MCI": "#DD8452", "AD": "#C44E52"}

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cols_show):
    ax = axes[i]
    for diag, color in COLOR_MAP.items():
        subset = df[df["dx_group"] == diag][col].dropna()
        if len(subset) > 1:
            ax.hist(subset, bins=25, alpha=0.5, color=color,
                    label=diag, density=True, edgecolor="none")
    ax.set_title(col, fontsize=9)
    ax.set_yticks([])
    if i == 0:
        ax.legend(fontsize=8)

# Masquer les axes vides si N_FEATURES_SHOW < 12
for j in range(N_FEATURES_SHOW, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution des features brutes par groupe (avant normalisation)",
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4 : Heatmap de corrélation entre features ────────────────────────────────
#
# Des corrélations fortes (> 0.9) indiquent des features redondantes
# → potentiel de réduction de dimension (PCA, sélection)
# On limite à 30 features pour la lisibilité

N_CORR = min(30, len(feature_cols))
corr_matrix = df[feature_cols[:N_CORR]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # masque triangle sup
sns.heatmap(corr_matrix,
            mask=mask,
            cmap="RdBu_r",
            center=0,
            vmin=-1, vmax=1,
            square=True,
            linewidths=0.3,
            cbar_kws={"shrink": 0.8},
            xticklabels=True,
            yticklabels=True,
            ax=ax)
ax.set_title(f"Corrélations entre les {N_CORR} premières features (brutes)",
             fontsize=13)
plt.xticks(fontsize=7, rotation=45, ha="right")
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

# Features les plus corrélées
corr_pairs = (
    corr_matrix.abs()
    .unstack()
    .sort_values(ascending=False)
    .drop_duplicates()
)
corr_high = corr_pairs[(corr_pairs > 0.85) & (corr_pairs < 1.0)]
print(f"Paires de features avec |corr| > 0.85 : {len(corr_high)}")
if len(corr_high) > 0:
    print(corr_high.head(10))

In [ ]:
# ── 5 : Valeurs manquantes (NaN) ─────────────────────────────────────────────
#
# FreeSurfer peut produire des NaN sur certaines ROI (échec de segmentation)
# On visualise la proportion de NaN par feature avant imputation

nan_pct = df[feature_cols].isnull().mean() * 100  # % NaN par feature
nan_pct = nan_pct[nan_pct > 0].sort_values(ascending=False)

n_ok  = (df[feature_cols].isnull().sum() == 0).sum()
n_tot = len(feature_cols)
print(f"Features sans aucun NaN : {n_ok} / {n_tot}")

if len(nan_pct) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, len(nan_pct) * 0.3)))
    nan_pct.plot.barh(ax=ax, color="#C44E52", edgecolor="none")
    ax.set_xlabel("% valeurs manquantes")
    ax.set_title("Features avec NaN (avant imputation)")
    plt.tight_layout()
    plt.show()
else:
    print("Aucun NaN détecté — dataset propre.")

---
## 🔧 Bloc 2 — Prétraitement & Corruption (SSL)

Deux étapes :
1. **Nettoyage** : suppression NaN, séparation features / labels
2. **Normalisation** : StandardScaler (moyenne=0, std=1) — indispensable pour un DAE
3. **Corruption** : stratégie SSL — on apprend à débruiter → représentation robuste

Deux types de bruit combinables :
- `masking_noise` : met aléatoirement `p` features à zéro
- `gaussian_noise` : ajoute N(0, σ) sur toutes les features

In [ ]:
# ── Séparation features / labels ──────────────────────────────────────────────

# Colonnes features = tout sauf la colonne de diagnostic
feature_cols = [c for c in df.columns if c != "dx_group"]

X_raw = df[feature_cols].values.astype(np.float32)   # (N, D)
y_raw = df["dx_group"].values                          # (N,)  — labels bruts

# ── Suppression des NaN ───────────────────────────────────────────────────────
# On remplace les NaN par la médiane de chaque feature
col_medians = np.nanmedian(X_raw, axis=0)
nan_mask    = np.isnan(X_raw)
X_raw[nan_mask] = np.take(col_medians, np.where(nan_mask)[1])

print(f"Features : {X_raw.shape}  |  NaN restants : {np.isnan(X_raw).sum()}")

In [ ]:
# ── Train / Validation / Test split ───────────────────────────────────────────
#
# Important : le DAE s'entraîne SANS les labels
#   - train : 70 %  → apprentissage SSL
#   - val   : 15 %  → monitoring de la loss
#   - test  : 15 %  → évaluation finale (scores anomalie + AUC)

X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X_raw, y_raw, test_size=0.30, random_state=SEED, stratify=y_raw
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train : {X_train_raw.shape}  |  Val : {X_val_raw.shape}  |  Test : {X_test_raw.shape}")

# ── Normalisation (fit sur train uniquement — pas de data leakage) ─────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val   = scaler.transform(X_val_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

INPUT_DIM = X_train.shape[1]   # dimension d'entrée du DAE
print(f"Input dim : {INPUT_DIM}")

In [ ]:
# ── Fonctions de corruption (stratégie SSL) ───────────────────────────────────
#
# Le DAE apprend à reconstruire x_propre à partir de x_bruité
# → la perte = MSE(reconstruction, x_propre)
# → les cerveaux pathologiques mal reconstruits = score d'anomalie élevé

MASK_PROB     = 0.30   # probabilité de masquer une feature (mettre à 0)
GAUSSIAN_STD  = 0.10   # écart-type du bruit gaussien additif
NOISE_TYPE    = "both" # "masking" | "gaussian" | "both"

def corrupt(x: torch.Tensor, noise_type: str = "both") -> torch.Tensor:
    """
    Applique une corruption SSL sur un batch de features.

    Args:
        x          : tensor (batch_size, input_dim) — données propres
        noise_type : type de bruit à appliquer

    Returns:
        x_noisy    : tensor (batch_size, input_dim) — données corrompues
    """
    x_noisy = x.clone()

    if noise_type in ("masking", "both"):
        # Masque binaire aléatoire : MASK_PROB % des features → 0
        mask = torch.bernoulli(torch.full_like(x, 1 - MASK_PROB))
        x_noisy = x_noisy * mask

    if noise_type in ("gaussian", "both"):
        # Bruit gaussien additif centré
        gaussian = torch.randn_like(x) * GAUSSIAN_STD
        x_noisy = x_noisy + gaussian

    return x_noisy

# ── Test visuel de la corruption ──────────────────────────────────────────────
sample = torch.tensor(X_train[:1])            # 1 sujet
corrupted = corrupt(sample, NOISE_TYPE)

plt.figure(figsize=(12, 3))
plt.plot(sample[0].numpy(),    label="Original", alpha=0.8, linewidth=1.5)
plt.plot(corrupted[0].numpy(), label="Corrompu", alpha=0.7, linewidth=1, linestyle="--")
plt.title("Exemple de corruption SSL sur 1 sujet")
plt.xlabel("Feature index")
plt.ylabel("Valeur normalisée")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── DataLoaders PyTorch ────────────────────────────────────────────────────────
#
# On ne stocke PAS le bruit ici : la corruption est appliquée à la volée
# dans la boucle d'entraînement → chaque epoch voit un bruit différent

BATCH_SIZE = 64

train_dataset = TensorDataset(torch.tensor(X_train))
val_dataset   = TensorDataset(torch.tensor(X_val))
test_dataset  = TensorDataset(torch.tensor(X_test))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Batches train : {len(train_loader)}  |  val : {len(val_loader)}  |  test : {len(test_loader)}")

---
## 🏗️ Bloc 3 — Architecture DAE (PyTorch)

Architecture **MLP symétrique** :

```
Encodeur :  D → 256 → 128 → 64   (espace latent)
Décodeur :  64 → 128 → 256 → D   (reconstruction)
```

Chaque couche contient : `Linear → BatchNorm → ReLU → Dropout`  
Sauf la dernière (décodeur) qui est `Linear` nu pour reconstruire les valeurs continues.

In [ ]:
# ── Architecture du Denoising Autoencoder ─────────────────────────────────────

class DenoisingAutoencoder(nn.Module):
    """
    DAE MLP symétrique pour données tabular FreeSurfer.

    Encodeur : input_dim → [hidden_dims] → latent_dim
    Décodeur : latent_dim → [hidden_dims reversed] → input_dim

    Args:
        input_dim   : nombre de features d'entrée (ex. 90)
        hidden_dims : liste des dimensions cachées (ex. [256, 128])
        latent_dim  : dimension de l'espace latent (ex. 64)
        dropout     : taux de dropout (ex. 0.2)
    """

    def __init__(self,
                 input_dim:   int,
                 hidden_dims: list = [256, 128],
                 latent_dim:  int  = 64,
                 dropout:     float = 0.2):

        super().__init__()

        # ── Bloc utilitaire : couche dense complète ────────────────────────────
        def dense_block(in_dim, out_dim):
            """Linear → BatchNorm1d → ReLU → Dropout"""
            return nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.BatchNorm1d(out_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

        # ── Encodeur : input → hidden → latent ────────────────────────────────
        enc_layers = []
        prev_dim   = input_dim
        for h_dim in hidden_dims:
            enc_layers.append(dense_block(prev_dim, h_dim))
            prev_dim = h_dim
        # Dernière couche encodeur : vers l'espace latent (pas de dropout ici)
        enc_layers.append(nn.Linear(prev_dim, latent_dim))
        self.encoder = nn.Sequential(*enc_layers)

        # ── Décodeur : latent → hidden (inversé) → output ─────────────────────
        dec_layers = []
        prev_dim   = latent_dim
        for h_dim in reversed(hidden_dims):
            dec_layers.append(dense_block(prev_dim, h_dim))
            prev_dim = h_dim
        # Dernière couche décodeur : reconstruction sans activation
        dec_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Renvoie la représentation latente z."""
        return self.encoder(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Renvoie la reconstruction x̂ depuis z."""
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Passe avant complète : x → encodeur → z → décodeur → x̂
        Note : la corruption est appliquée AVANT l'appel à forward(),
        dans la boucle d'entraînement.
        """
        z    = self.encode(x)
        x_hat = self.decode(z)
        return x_hat


# ── Instanciation ──────────────────────────────────────────────────────────────
model = DenoisingAutoencoder(
    input_dim   = INPUT_DIM,
    hidden_dims = [256, 128],
    latent_dim  = 64,
    dropout     = 0.2
).to(DEVICE)

# Résumé du modèle
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParamètres entraînables : {total_params:,}")

---
## 🔁 Bloc 4 — Entraînement SSL

**Cycle SSL à chaque batch** :
1. `x_clean` = batch original normalisé
2. `x_noisy` = `corrupt(x_clean)` → bruit appliqué à la volée
3. `x_hat`   = `model(x_noisy)` → reconstruction
4. `loss`    = `MSE(x_hat, x_clean)` → on veut retrouver le signal propre
5. Backpropagation + Adam

> Le modèle apprend la **distribution normale** des cerveaux CN.  
> Les cerveaux AD s'en écartent → erreur de reconstruction plus élevée.

In [ ]:
# ── Hyperparamètres d'entraînement ────────────────────────────────────────────

N_EPOCHS    = 100      # nombre d'epochs (augmenter si overfitting non observé)
LR          = 1e-3     # learning rate Adam
WEIGHT_DECAY = 1e-5    # régularisation L2 légère

# ── Loss et optimiseur ────────────────────────────────────────────────────────
criterion = nn.MSELoss()                        # MSE(x̂, x_clean)
optimizer = optim.Adam(
    model.parameters(),
    lr           = LR,
    weight_decay = WEIGHT_DECAY
)

# ── Scheduler : réduit le LR si la val_loss stagne ───────────────────────────
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)
# Note : verbose=True retiré (supprimé dans PyTorch >= 2.2)
# Le LR courant est loggué dans la colonne LR du tableau ci-dessous

# ── Early stopping (simple) ───────────────────────────────────────────────────
PATIENCE_ES    = 20      # arrêt si pas d'amélioration pendant N epochs
best_val_loss  = float("inf")
epochs_no_imp  = 0
best_weights   = None    # on sauvegarde le meilleur état

In [ ]:
# ── Boucle d'entraînement principale ──────────────────────────────────────────

history = {"train_loss": [], "val_loss": []}

print(f"{'Epoch':>6} | {'Train Loss':>12} | {'Val Loss':>10} | {'LR':>10}")
print("-" * 50)

for epoch in range(1, N_EPOCHS + 1):

    # ── Phase entraînement ────────────────────────────────────────────────────
    model.train()
    train_losses = []

    for (x_batch,) in train_loader:
        x_clean = x_batch.to(DEVICE)           # données propres
        x_noisy = corrupt(x_clean, NOISE_TYPE)  # corruption SSL

        optimizer.zero_grad()
        x_hat = model(x_noisy)                  # reconstruction depuis bruité
        loss  = criterion(x_hat, x_clean)        # erreur vs données propres
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # ── Phase validation ──────────────────────────────────────────────────────
    model.eval()
    val_losses = []

    with torch.no_grad():
        for (x_batch,) in val_loader:
            x_clean = x_batch.to(DEVICE)
            x_noisy = corrupt(x_clean, NOISE_TYPE)
            x_hat   = model(x_noisy)
            val_losses.append(criterion(x_hat, x_clean).item())

    # ── Logging ───────────────────────────────────────────────────────────────
    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    scheduler.step(val_loss)

    # Affichage toutes les 10 epochs
    if epoch % 10 == 0 or epoch == 1:
        print(f"{epoch:>6} | {train_loss:>12.6f} | {val_loss:>10.6f} | {current_lr:>10.2e}")

    # ── Early stopping ────────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_imp = 0
        # Sauvegarde des meilleurs poids
        best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_imp += 1
        if epochs_no_imp >= PATIENCE_ES:
            print(f"\nEarly stopping à l'epoch {epoch} (meilleure val_loss : {best_val_loss:.6f})")
            break

# Restauration des meilleurs poids
if best_weights:
    model.load_state_dict(best_weights)
    print(f"\nPoids restaurés — best val_loss : {best_val_loss:.6f}")

In [ ]:
# ── Courbe de loss (train vs validation) ──────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history["train_loss"], label="Train loss", color="#4C72B0", linewidth=2)
ax.plot(history["val_loss"],   label="Val loss",   color="#DD8452", linewidth=2, linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Courbe d'entraînement DAE-SSL")
ax.legend()
ax.set_yscale("log")   # log scale pour mieux voir la convergence
plt.tight_layout()
plt.show()

print(f"Train loss finale : {history['train_loss'][-1]:.6f}")
print(f"Val   loss finale : {history['val_loss'][-1]:.6f}")

---
## 🔴 Bloc 5 — Score d'anomalie

**Principe** : après entraînement sur cerveaux normaux (surtout CN),  
le DAE reconstruit bien les CN mais mal les cerveaux pathologiques.

**Score anomalie** par sujet =  
$$\text{anomaly\_score}(x) = \frac{1}{D} \sum_{d=1}^{D} (x_d - \hat{x}_d)^2$$

→ Plus le score est élevé, plus le sujet est anormal.

In [ ]:
# ── Calcul des scores d'anomalie sur le set de test ───────────────────────────

model.eval()

all_scores = []   # erreur de reconstruction par sujet
all_latents = []  # vecteurs latents z (pour t-SNE)

with torch.no_grad():
    for (x_batch,) in test_loader:
        x_clean = x_batch.to(DEVICE)

        # Reconstruction sans bruit (inférence propre)
        z     = model.encode(x_clean)
        x_hat = model.decode(z)

        # Erreur par sujet : MSE sur toutes les features
        # shape (batch_size, input_dim) → mean sur dim=1 → (batch_size,)
        errors = ((x_clean - x_hat) ** 2).mean(dim=1)

        all_scores.append(errors.cpu().numpy())
        all_latents.append(z.cpu().numpy())

anomaly_scores = np.concatenate(all_scores)   # (N_test,)
latent_vecs    = np.concatenate(all_latents)  # (N_test, latent_dim)

print(f"Scores calculés pour {len(anomaly_scores)} sujets")
print(f"Score min : {anomaly_scores.min():.4f}  |  max : {anomaly_scores.max():.4f}  |  moy : {anomaly_scores.mean():.4f}")

In [ ]:
# ── Seuillage : définition du seuil d'anomalie ────────────────────────────────
#
# Stratégie : percentile 90 des scores CN sur le train set
# (approche non-supervisée — aucun label utilisé pour fixer le seuil)

# Calcul des scores sur le train set pour établir le seuil
model.eval()
train_scores_list = []

with torch.no_grad():
    for (x_batch,) in train_loader:
        x_clean = x_batch.to(DEVICE)
        z       = model.encode(x_clean)
        x_hat   = model.decode(z)
        errors  = ((x_clean - x_hat) ** 2).mean(dim=1)
        train_scores_list.append(errors.cpu().numpy())

train_scores = np.concatenate(train_scores_list)

# Seuil = percentile 90 (tunable)
THRESHOLD_PERCENTILE = 90
THRESHOLD = np.percentile(train_scores, THRESHOLD_PERCENTILE)
print(f"Seuil anomalie (p{THRESHOLD_PERCENTILE}) : {THRESHOLD:.4f}")

# Prédictions binaires : 1 = anomalie, 0 = normal
y_pred_binary = (anomaly_scores > THRESHOLD).astype(int)

# Vérité terrain binaire : 1 = AD ou MCI, 0 = CN
y_true_binary = (y_test != "CN").astype(int)

print(f"\nPrédictions : {y_pred_binary.sum()} anomalies détectées sur {len(y_pred_binary)} sujets")

---
## 📊 Bloc 6 — Visualisations & Évaluation

4 visualisations :
1. **Distribution des scores** par groupe CN / MCI / AD
2. **Boxplot** des scores par diagnostic
3. **t-SNE** de l'espace latent coloré par groupe
4. **Rapport de classification** + AUC ROC

In [ ]:
# ── 1 & 2 : Distribution et boxplot des scores ────────────────────────────────

# DataFrame résultat
df_results = pd.DataFrame({
    "anomaly_score" : anomaly_scores,
    "diagnosis"     : y_test,
    "is_anomaly"    : y_pred_binary
})

COLOR_MAP = {"CN": "#4C72B0", "MCI": "#DD8452", "AD": "#C44E52"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Distribution KDE ──────────────────────────────────────────────────────────
ax = axes[0]
for diag, color in COLOR_MAP.items():
    subset = df_results[df_results["diagnosis"] == diag]["anomaly_score"]
    if len(subset) > 0:
        subset.plot.kde(ax=ax, label=diag, color=color, linewidth=2)

ax.axvline(THRESHOLD, color="black", linestyle="--", linewidth=1.5, label=f"Seuil (p{THRESHOLD_PERCENTILE})")
ax.set_xlabel("Score d'anomalie (MSE reconstruction)")
ax.set_ylabel("Densité")
ax.set_title("Distribution des scores par groupe")
ax.legend()

# ── Boxplot ───────────────────────────────────────────────────────────────────
ax = axes[1]
order = [g for g in ["CN", "MCI", "AD"] if g in df_results["diagnosis"].values]
palette = {k: v for k, v in COLOR_MAP.items() if k in order}

sns.boxplot(data=df_results, x="diagnosis", y="anomaly_score",
            order=order, palette=palette, ax=ax)
ax.axhline(THRESHOLD, color="black", linestyle="--", linewidth=1.5, label=f"Seuil")
ax.set_xlabel("Diagnostic")
ax.set_ylabel("Score d'anomalie")
ax.set_title("Scores par groupe (boxplot)")
ax.legend()

plt.suptitle("DAE-SSL : Scores d'anomalie — OASIS-3", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3 : t-SNE de l'espace latent ──────────────────────────────────────────────
#
# Réduction 64-dim → 2-dim pour visualiser les clusters dans z
# Si les CN se regroupent séparément des AD : la représentation SSL est pertinente

print("Calcul t-SNE (peut prendre ~30s)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
latent_2d = tsne.fit_transform(latent_vecs)   # (N_test, 2)

df_tsne = pd.DataFrame({
    "x"        : latent_2d[:, 0],
    "y"        : latent_2d[:, 1],
    "diagnosis": y_test
})

fig, ax = plt.subplots(figsize=(8, 6))
for diag, color in COLOR_MAP.items():
    subset = df_tsne[df_tsne["diagnosis"] == diag]
    if len(subset) > 0:
        ax.scatter(subset["x"], subset["y"],
                   label=diag, color=color, alpha=0.7, s=25, edgecolors="none")

ax.set_title("t-SNE de l'espace latent z (64-dim → 2-dim)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(title="Diagnostic")
plt.tight_layout()
plt.show()

In [ ]:
# ── 4 : Métriques d'évaluation ────────────────────────────────────────────────
#
# Rappel : ces métriques ne servent QU'à évaluer — les labels n'ont jamais
# été utilisés pendant l'entraînement (c'est le principe du SSL)

print("=" * 50)
print("RAPPORT DE CLASSIFICATION (CN normal vs MCI/AD anomalie)")
print("=" * 50)
print(classification_report(y_true_binary, y_pred_binary,
                             target_names=["Normal (CN)", "Anomalie (MCI/AD)"]))

# AUC-ROC : utilise le score continu (plus informatif que le binaire)
if len(np.unique(y_true_binary)) > 1:   # vérification qu'il y a les deux classes
    auc = roc_auc_score(y_true_binary, anomaly_scores)
    print(f"AUC-ROC : {auc:.4f}")
    print("(0.5 = aléatoire, 1.0 = parfait)")

# Score moyen par groupe (diagnostic)
print("\nScore moyen par groupe :")
print(df_results.groupby("diagnosis")["anomaly_score"].agg(["mean", "std", "count"]))

In [ ]:
# ── Sauvegarde du modèle ──────────────────────────────────────────────────────
#
# Sauvegarde de l'état du modèle + du scaler pour réutilisation
# sans réentraîner (ex. sur nouveaux sujets ADNI)

import pickle

# Poids PyTorch
torch.save(model.state_dict(), "/content/dae_ssl_oasis3.pt")

# Scaler sklearn
with open("/content/scaler_oasis3.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Modèle sauvegardé : /content/dae_ssl_oasis3.pt")
print("Scaler sauvegardé : /content/scaler_oasis3.pkl")

# ── Chargement (exemple) ──────────────────────────────────────────────────────
# model_loaded = DenoisingAutoencoder(INPUT_DIM, [256, 128], 64)
# model_loaded.load_state_dict(torch.load("/content/dae_ssl_oasis3.pt"))
# model_loaded.eval()

---
## ➡️ Prochaines étapes

| Étape | Action |
|-------|--------|
| Données réelles | Télécharger OASIS-3 FreeSurfer stats → passer `USE_SYNTHETIC = False` |
| Fine-tuning | Geler l'encodeur, ajouter une tête de classification CN/MCI/AD |
| ADNI | Transférer le modèle pré-entraîné sur OASIS vers ADNI |
| 3D | Remplacer le MLP par un CNN 3D pour travailler sur les NIfTI bruts |
| Interprétabilité | SHAP sur les features avec score d'anomalie élevé |

---
*Pipeline DAE-SSL — Thèse : Brain Health Assessment from Neuroimaging Data*